<a href="https://colab.research.google.com/github/Yukselendincer/datasceinceproject/blob/main/FrudDetected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd

# Google Drive'ı bağla
drive.mount('/content/drive')


path = "/content/creditcard.csv" # Corrected path based on available files
df = pd.read_csv(path)

# Veriye hızlı bir bakış
print(f"Veri Seti Boyutu: {df.shape}")
df.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Dolandırıcılık oranını görselleştir
plt.figure(figsize=(8,5))
sns.countplot(x='Class', data=df)
plt.title('İşlem Dağılımı (0: Normal, 1: Fraud)')
plt.show()

fraud_rate = df['Class'].value_counts(normalize=True)[1] * 100
print(f"Dolandırıcılık Oranı: %{fraud_rate:.2f}") # Genelde %0.17 civarıdır.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, average_precision_score

# Veriyi Hazırla
X = df.drop('Class', axis=1)
y = df['Class']

# Veriyi Eğitim ve Test olarak ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE ile sentetik veri artırımı (Sadece eğitim setine!)
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# Model: XGBoost (Hızlı ve Güçlü)
model = XGBClassifier(scale_pos_weight=1, learning_rate=0.1, n_estimators=100)
model.fit(X_train_res, y_train_res)

# Tahmin
y_pred = model.predict(X_test)

In [ ]:
print("--- Fraud Tespit Raporu ---")
print(classification_report(y_test, y_pred))

# Hassasiyet-Duyarlılık Puanı
auprc = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"AUPRC Skoru: {auprc:.4f}")